# TutorBuddy
### AI-Powered Tutor for Grade 4–8 Students

**TutorBuddy** is an intelligent tutoring agent built for the Kaggle 5-Day AI Intensive capstone. It helps students in grades 4–8 with homework questions, lesson summaries, quiz generation, and worksheet OCR — supporting both English and Urdu.

### Why It Matters
Many students in under-resourced classrooms lack access to one-on-one tutoring. TutorBuddy provides:
- **Instant answers** grounded in a curated curriculum (RAG)
- **Lesson summaries** in simple language
- **Practice quizzes** for self-assessment
- **OCR** to digitize handwritten or printed worksheets

The stack: **Gemini 1.5 Flash** (LLM), **all-MiniLM-L6-v2** (embeddings), **FAISS** (vector search), **Tesseract** (OCR), and **Gradio** (UI).

## 1. Setup

Install dependencies, import libraries, and configure the Gemini API key.

> **On Kaggle:** Use `UserSecretsClient` to fetch the key.  
> **Locally:** Set `GEMINI_API_KEY` in your environment or a `.env` file.

In [ ]:
# Install dependencies (Kaggle: run this in the first cell)
!pip install -q faiss-cpu sentence-transformers google-genai python-dotenv gradio opencv-python-headless pytesseract pillow pandas scikit-learn matplotlib

In [ ]:
import json
import os
import re
import csv
import pickle
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- API Key - auto-detect Kaggle vs local ----
if Path("/kaggle").exists():
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"] = api_key
else:
    from dotenv import load_dotenv
    load_dotenv()


# ---- Kaggle / Local path resolution ----
KAGGLE_BASE = Path("/kaggle/input/tutorbuddy-curriculum")

def data_path(relative: str) -> Path:
    """Resolve a data file path for Kaggle vs local. Output files stay local."""
    if KAGGLE_BASE.exists():
        rel = relative.removeprefix("data/")
        return KAGGLE_BASE / rel
    return Path(relative)
print("Setup complete. API key present:", bool(os.getenv("GEMINI_API_KEY")))

## 2. Data — Curriculum Notes

The knowledge base contains **30 short educational notes** across Math, Science, and English for grades 4–8. Each note has a `topic`, `subject`, `grade`, and `content` field.

We also have **10 evaluation question-answer pairs** used to measure retrieval and answer quality.

In [ ]:
# Load and display curriculum notes
with open(data_path("data/curriculum/notes.json")) as f:
    notes = json.load(f)

df_notes = pd.DataFrame(notes)
print(f"Total notes: {len(df_notes)}")
print(f"Subjects: {df_notes['subject'].value_counts().to_dict()}")
display(df_notes.head(5))

In [ ]:
# Load and display evaluation questions
with open(data_path("data/eval/eval_questions.json")) as f:
    eval_questions = json.load(f)

df_eval = pd.DataFrame(eval_questions)
print(f"Total eval questions: {len(df_eval)}")
display(df_eval)

## 3. Retrieval — FAISS + Sentence Transformers

`CurriculumRetriever` embeds all 30 notes with `all-MiniLM-L6-v2` and stores them in a FAISS `IndexFlatL2`. When a question comes in, it finds the 3 most relevant notes and returns them as context for the LLM.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer


class CurriculumRetriever:
    def __init__(self, notes_path=str(data_path("data/curriculum/notes.json")),
                 index_path="data/faiss_index.bin"):
        self.notes_path = Path(notes_path)
        self.index_path = Path(index_path)
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.notes: list[dict] = []
        self.index: faiss.Index | None = None
        self._load_or_build_index()

    def _load_notes(self):
        with open(self.notes_path) as f:
            return json.load(f)

    def _build_index(self):
        self.notes = self._load_notes()
        texts = [note["content"] for note in self.notes]
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        dim = embeddings.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings.astype(np.float32))
        return index

    def _save_index(self):
        self.index_path.parent.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(self.index_path))
        meta_path = self.index_path.with_suffix(".pkl")
        with open(meta_path, "wb") as f:
            pickle.dump(self.notes, f)

    def _load_index(self):
        meta_path = self.index_path.with_suffix(".pkl")
        if self.index_path.exists() and meta_path.exists():
            try:
                self.index = faiss.read_index(str(self.index_path))
                with open(meta_path, "rb") as f:
                    self.notes = pickle.load(f)
                return True
            except Exception:
                return False
        return False

    def _load_or_build_index(self):
        if not self._load_index():
            self.index = self._build_index()
            self._save_index()

    def retrieve(self, query: str, top_k: int = 3) -> str:
        query_emb = self.model.encode([query], convert_to_numpy=True).astype(np.float32)
        distances, indices = self.index.search(query_emb, top_k)
        results = []
        for i in indices[0]:
            note = self.notes[i]
            results.append(
                f"Topic: {note['topic']}\nGrade: {note['grade']}\n{note['content']}"
            )
        return "\n\n".join(results)

In [ ]:
# Build and test the retriever
retriever = CurriculumRetriever()
print("FAISS index size:", retriever.index.ntotal)

query = "What is photosynthesis?"
result = retriever.retrieve(query)
print(f"\nQuery: {query}\n")
print(result)

## 4. Agent — TutorAgent with Gemini

`TutorAgent` wraps the retriever and Gemini 2.5 Flash. For each question, it:
1. Retrieves relevant curriculum context via FAISS
2. Builds a prompt with a safety system instruction
3. Calls Gemini and returns a student-friendly answer

In [ ]:
from google import genai
from google.genai import types as genai_types

_SYSTEM_INSTRUCTION = (
    "You are TutorBuddy, a friendly AI tutor for Grade 4–8 students. "
    "Always use simple, clear language appropriate for children aged 9–14. "
    "Never produce adult content, harmful content, or anything unrelated to education. "
    "Stay focused on the student's question only."
)


class TutorAgent:
    def __init__(self, retriever: CurriculumRetriever):
        self.retriever = retriever
        self.client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
        self.model = "gemini-2.5-flash"

    def _build_system_instruction(self, language: str = "english") -> str:
        instruction = _SYSTEM_INSTRUCTION
        if language.lower() == "urdu":
            instruction += " Respond in Urdu."
        return instruction

    def _generate(self, prompt: str, system: str) -> str:
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
            config=genai_types.GenerateContentConfig(
                system_instruction=system,
            ),
        )
        return response.text

    def answer_question(self, question: str, language: str = "english") -> str:
        try:
            context = self.retriever.retrieve(question)
        except Exception as e:
            return f"Could not search the curriculum: {e}"
        system = self._build_system_instruction(language)
        prompt = (
            f"Here is some relevant information:\n{context}\n\n"
            f"Answer the following question in a friendly, simple way:\n{question}"
        )
        try:
            return self._generate(prompt, system)
        except Exception as e:
            err = str(e).lower()
            if "api key" in err or "not found" in err or "permission" in err or "access" in err:
                return "API key is missing or invalid."
            if "safety" in err or "blocked" in err:
                return "Blocked by safety filters. Try rephrasing."
            if "quota" in err or "rate" in err or "limit" in err or "429" in err:
                return "Rate limit reached. Please wait."
            return f"Gemini API error: {e}"

    def summarize_lesson(self, text: str, language: str = "english") -> str:
        if not text.strip():
            return "Please provide some lesson text."
        system = self._build_system_instruction(language)
        prompt = (
            "Summarize the following lesson in simple words that a Grade 4–8 student can understand. "
            f"Keep it short and clear:\n\n{text}"
        )
        try:
            return self._generate(prompt, system)
        except Exception as e:
            return f"Error: {e}"

    def generate_quiz(self, topic: str, num_questions: int = 3) -> str:
        if not topic.strip():
            return "Please provide a topic."
        try:
            context = self.retriever.retrieve(topic)
        except Exception as e:
            return f"Search error: {e}"
        system = self._build_system_instruction("english")
        prompt = (
            f"Here is some relevant information:\n{context}\n\n"
            f"Create a {num_questions}-question multiple-choice quiz on the topic '{topic}'. "
            "For each question, provide four options (A, B, C, D), the correct answer, "
            "and a short explanation. Use simple language."
        )
        try:
            return self._generate(prompt, system)
        except Exception as e:
            return f"Error: {e}"

In [ ]:
# Create the agent and test with a few questions
agent = TutorAgent(retriever)

questions = [
    "What is a fraction?",
    "What is photosynthesis?",
    "What is a noun?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print("-" * 60)
    answer = agent.answer_question(q)
    print(f"A: {answer}\n")

## 5. OCR Demo — Worksheet Text Extraction

TutorBuddy can extract text from images using **Tesseract OCR** with OpenCV preprocessing (grayscale + OTSU threshold). This allows students to take a photo of a worksheet and get the text digitized.

In [ ]:
import cv2
import pytesseract
from PIL import Image


def extract_text_from_image(image_path: str) -> str:
    try:
        img = Image.open(image_path).convert("RGB")
        img = np.array(img)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        text = pytesseract.image_to_string(thresh)
        cleaned = "\n".join(line.strip() for line in text.splitlines() if line.strip())
        return cleaned
    except Exception as e:
        print(e)
        return "Could not read image. Please try a clearer photo."


# Demo: extract from a sample image if available
demo_image = str(data_path("assets/test_worksheet.png"))
if Path(demo_image).exists():
    text = extract_text_from_image(demo_image)
    print("Extracted text:")
    print(text)
else:
    print(f"No test image found at {demo_image}. Upload a worksheet image to test OCR.")

## 6. Quiz Generation

TutorBuddy can generate multiple-choice quizzes on any topic. It uses the retriever to find relevant context and asks Gemini to create questions with options, answers, and explanations.

In [ ]:
# Generate a sample quiz
topic = "fractions"
print(f"Generating a 2-question quiz on '{topic}'...\n")
quiz = agent.generate_quiz(topic, num_questions=2)
print(quiz)

## 7. Gradio UI — Interactive Demo

TutorBuddy comes with a full **Gradio** interface featuring four tabs:
- **Ask a Question** — type any question, pick English/Urdu, get an answer
- **Summarize Lesson** — paste a lesson, get a simplified summary
- **Generate Quiz** — pick a topic, get an interactive quiz with radio buttons
- **Worksheet OCR** — upload an image, extract text, send it to the Ask tab

Launch the cell below to get a **public shareable link** (valid for 72 hours). Click the URL to interact with TutorBuddy live.

In [ ]:
# Launch the Gradio UI with a public share link
import sys
from pathlib import Path
import gradio as gr

# CurriculumRetriever and TutorAgent are inlined above - no src imports needed
retriever = CurriculumRetriever(
    notes_path=str(data_path("data/curriculum/notes.json")),
)
agent = TutorAgent(retriever)

def answer(question, language):
    lang = "urdu" if language == "Urdu" else "english"
    return agent.answer_question(question, lang)

def summarize(text):
    return agent.summarize_lesson(text)

def quiz_text(topic, num):
    return agent.generate_quiz(topic, num)

def ocr(image_path):
    if not image_path:
        return ""
    return extract_text_from_image(image_path)


with gr.Blocks(title="TutorBuddy", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# TutorBuddy\n"
        "### Your friendly AI tutor for Grade 4–8 students\n"
        "Ask questions, summarize lessons, generate quizzes, and extract text from worksheets."
    )

    with gr.Tab("Ask a Question"):
        q_in = gr.Textbox(label="Your Question", placeholder="e.g. What is the water cycle?")
        lang_dd = gr.Dropdown(["English", "Urdu"], label="Language", value="English")
        gr.Button("Ask", variant="primary").click(
            answer, inputs=[q_in, lang_dd], outputs=gr.Textbox(label="Answer", lines=6)
        )

    with gr.Tab("Summarize Lesson"):
        l_in = gr.Textbox(label="Lesson Text", lines=8, placeholder="Paste the lesson here...")
        gr.Button("Summarize", variant="primary").click(
            summarize, inputs=[l_in], outputs=gr.Textbox(label="Summary", lines=6)
        )

    with gr.Tab("Generate Quiz"):
        t_in = gr.Textbox(label="Topic", placeholder="e.g. fractions")
        n_slider = gr.Slider(1, 5, value=3, step=1, label="Number of Questions")
        gr.Button("Generate Quiz", variant="primary").click(
            quiz_text, inputs=[t_in, n_slider], outputs=gr.Textbox(label="Quiz", lines=10)
        )

    with gr.Tab("Worksheet OCR"):
        img_in = gr.Image(type="filepath", label="Upload Worksheet Image")
        gr.Button("Extract Text", variant="primary").click(
            ocr, inputs=[img_in], outputs=gr.Textbox(label="Extracted Text", lines=6)
        )

demo.queue()
demo.launch(share=True, server_port=7860)
print("\nClick the URL above to interact with TutorBuddy!")

## 8. Evaluation

We evaluate TutorBuddy by running all 10 eval questions and measuring **keyword overlap** (Jaccard similarity) between the expected answer and the model's response. This is a simple, reproducible metric.

We also track **response length** and compute a **pass rate** (score ≥ 0.3).

In [ ]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "to", "of", "in", "for",
    "on", "with", "at", "by", "from", "as", "into", "through", "during",
    "before", "after", "above", "below", "between", "and", "but", "or",
    "nor", "not", "so", "yet", "both", "either", "neither", "each",
    "every", "all", "any", "few", "more", "most", "other", "some", "such",
    "no", "only", "own", "same", "than", "too", "very", "just", "because",
    "about", "if", "then", "else", "when", "where", "why", "how", "which",
    "who", "whom", "what", "this", "that", "these", "those", "it", "its",
    "you", "your", "they", "their", "we", "our", "i", "me", "my", "he",
    "she", "him", "her", "his",
}


def tokenize(text: str) -> set[str]:
    tokens = re.findall(r"[a-zA-Z\u0600-\u06FF]+", text.lower())
    return {t for t in tokens if t not in STOPWORDS and len(t) > 1}


def keyword_overlap(expected: str, actual: str) -> float:
    exp_tokens = tokenize(expected)
    act_tokens = tokenize(actual)
    if not exp_tokens or not act_tokens:
        return 0.0
    return len(exp_tokens & act_tokens) / len(exp_tokens | act_tokens)

In [ ]:
# Run evaluation on all 10 questions
results = []
for q in eval_questions:
    question = q["question"]
    expected = q["expected_answer"]
    subject = q["subject"]

    response = agent.answer_question(question)
    score = keyword_overlap(expected, response)
    length = len(response.split())

    results.append({
        "question": question,
        "subject": subject,
        "score": round(score, 3),
        "length": length,
    })
    print(f"[{subject:8}] score={score:.3f}  len={length:3}  {question}")
    time.sleep(1)

df_results = pd.DataFrame(results)
print("\n" + "=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
avg_score = df_results["score"].mean()
pass_rate = (df_results["score"] >= 0.3).mean()
avg_length = df_results["length"].mean()
print(f"  Avg keyword overlap: {avg_score:.3f}")
print(f"  Pass rate (>=0.3):   {pass_rate:.1%}")
print(f"  Avg response length: {avg_length:.1f} words")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score by subject
df_results.boxplot(column="score", by="subject", ax=axes[0])
axes[0].set_title("Keyword Overlap Score by Subject")
axes[0].set_ylabel("Score (Jaccard)")
axes[0].set_xlabel("")

# Score distribution
axes[1].barh(range(len(df_results)), df_results["score"].values, color="steelblue")
axes[1].set_yticks(range(len(df_results)))
axes[1].set_yticklabels(df_results["subject"].values + ": " + df_results["question"].str[:25] + "...")
axes[1].set_xlabel("Score")
axes[1].set_title("Per-Question Scores")
axes[1].axvline(0.3, color="red", linestyle="--", label="Pass threshold")
axes[1].legend()
plt.tight_layout()
plt.show()

# Save results to CSV
results_path = Path("results.csv") if KAGGLE_BASE.exists() else Path("data/eval/results.csv")
results_path.parent.mkdir(parents=True, exist_ok=True)
df_results.to_csv(results_path, index=False)
print(f"Results saved to {results_path}")

## 9. Conclusion

### What Works
- **RAG pipeline** reliably retrieves relevant curriculum notes for math, science, and English questions
- **Gemini 2.5 Flash** produces clear, age-appropriate answers with proper safety guardrails
- **Keyword overlap evaluation** provides a repeatable baseline metric (pass rate at ≥0.3 threshold)
- **OCR pipeline** successfully extracts text from clean worksheet images
- **Quiz generation** creates structured multiple-choice questions with explanations
- **Urdu support** works at the prompt level without needing a separate translation model

### What Could Be Improved
- **Better evaluation metrics**: Keyword overlap is simplistic. Future work could use semantic similarity (e.g., sentence-BERT cosine) or LLM-as-judge scoring for more nuanced quality measurement
- **Curriculum coverage**: 30 notes is a minimal knowledge base. Scaling to hundreds of notes would improve answer quality and breadth
- **OCR robustness**: Tesseract struggles with handwriting, low-light photos, and complex layouts. A cloud OCR API (Google Document AI) would be more reliable
- **Quiz interactivity**: The Gradio UI already supports interactive quiz checking, but the notebook demo uses the text-only generation path
- **Student personalization**: No user modeling yet — adapting explanations to each student's grade level and past performance would make TutorBuddy truly adaptive
- **Multi-turn conversations**: Currently each question is independent. Adding chat memory would allow follow-up questions

---
*Built for the Kaggle 5-Day Generative AI Intensive — Capstone Project*